In [0]:
cubeserviceofficetxn_path=dbutils.widgets.get("cubeserviceofficetxn_path")
data_path=dbutils.widgets.get("date_path")
fact_revenue_path=dbutils.widgets.get("fact_revenue_path")

In [0]:
spark.sql(
    f"""
-- STEP 1: CALCULATE WEEK ENDING DATE

DECLARE OR REPLACE WeekendingDate STRING;
"""
)

spark.sql(
    f"""
SET VAR WeekendingDate = (
    SELECT REPLACE(
        CAST(DATE_SUB(WeekEndingDate, 7) AS STRING),
        '-',
        ''
    )
    FROM {data_path}
    WHERE CalendarDate = CURRENT_DATE()
);
"""
)

spark.sql(
    f"""
    TRUNCATE TABLE {fact_revenue_path} ;

    """ 
)


In [0]:
spark.sql(
    f"""
    INSERT INTO {fact_revenue_path} (
        week_ending_date_key,
        visit_id,
        service_date_key,
        source_system_key,
        office_key,
        payor_key,
        client_key,
        approval_key,
        current_period_total_billed
    )
    SELECT 
        WeekEndingDateKey AS week_ending_date_key,
        visitID AS visit_id,
        ServiceDateKey AS service_date_key,
        SourceSystemKey AS source_system_key,
        OfficeKey AS office_key,
        PayorKey AS payor_key,
        ClientKey AS client_key,
        ApprovalKey AS approval_key,
        CurrentPeriodTotalBilled AS current_period_total_billed
    FROM {cubeserviceofficetxn_path}
    WHERE LEFT(CAST(WeekEndingDateKey AS STRING), 4) >= '2020'
      AND ApprovalKey = 10;
    """
)